In [0]:
%pip install xlrd

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd, re
path = "/Volumes/workspace/default/test/Road-Accident-Safety-Data-Guide.xls"
xl = pd.ExcelFile(path)
for s in xl.sheet_names:
    df = xl.parse(s)
    if {"code","label"} <= set(df.columns):
        df["code"] = pd.to_numeric(df["code"], errors="coerce")
        df = df.dropna(subset=["code"])
        df["code"] = df["code"].astype("int64")
        df["label"] = df["label"].astype(str)
        name = "dim_" + re.sub(r"\W+", "_", s.strip().lower()).strip("_")
        spark.createDataFrame(df).write.mode("overwrite").saveAsTable(f"workspace.default.{name}")
        print("created", name, len(df))
    else:
        print("skipped", s, list(df.columns))

In [0]:
%sql
SHOW TABLES IN workspace.default LIKE 'dim_*';

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.accidents_clean AS
SELECT a.Accident_Index,
       a.Date,
       YEAR(a.Date) AS year,
       HOUR(a.Time) AS hour,
       a.Latitude,
       a.Longitude,
       a.Speed_limit,
       a.Number_of_Vehicles,
       a.Number_of_Casualties,
       a.Accident_Severity AS severity_code,
       sev.label AS severity,
       w.label   AS weather,
       rt.label  AS road_type,
       l.label   AS light_conditions,
       d.label   AS day_of_week
FROM workspace.default.accidents_0514 a
LEFT JOIN workspace.default.dim_accident_severity sev ON a.Accident_Severity = sev.code
LEFT JOIN workspace.default.dim_weather w             ON a.Weather_Conditions = w.code
LEFT JOIN workspace.default.dim_road_type rt          ON a.Road_Type = rt.code
LEFT JOIN workspace.default.dim_light_conditions l    ON a.Light_Conditions = l.code
LEFT JOIN workspace.default.dim_day_of_week d         ON a.Day_of_Week = d.code
WHERE a.Latitude IS NOT NULL AND a.Longitude IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.hotspot_detail AS
SELECT ROUND(Latitude,2) AS lat,
       ROUND(Longitude,2) AS lon,
       year, weather, road_type,
       COUNT(*) AS accidents,
       SUM(CASE WHEN severity_code = 1 THEN 1 ELSE 0 END) AS fatal,
       SUM(CASE WHEN severity_code = 2 THEN 1 ELSE 0 END) AS serious,
       SUM(CASE severity_code WHEN 1 THEN 10 WHEN 2 THEN 3 ELSE 1 END) AS risk_score
FROM workspace.default.accidents_clean
GROUP BY 1,2,3,4,5;

In [0]:
%sql
SELECT (SELECT SUM(accidents) FROM workspace.default.hotspot_detail) AS hotspot_total,
       (SELECT COUNT(*) FROM workspace.default.accidents_clean) AS clean_total;

In [0]:
%sql
   SELECT * FROM workspace.default.hotspot_detail